# EDA Turnaround Dataset

- `data/1_raw/flight_list`
- `data/3_feature_engineered/flight_list`
- `data/4_final/model_input_merged.csv`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 12,
        "axes.edgecolor": "#4A4A4A",
        "grid.color": "#D9D9D9",
        "grid.linewidth": 0.6,
        "grid.alpha": 0.7,
        "savefig.dpi": 300,
    }
)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

COLORS = {
    "primary": "#16324F",
    "secondary": "#3A6EA5",
    "accent": "#B85C38",
    "highlight": "#2A9D8F",
    "gold": "#C79A2B",
    "muted": "#6C757D",
}


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data").exists() and (candidate / "source").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current notebook location")


REPO_ROOT = find_repo_root(Path.cwd())
RAW_FLIGHT_DIR = REPO_ROOT / "data" / "1_raw" / "flight_list"
FEATURE_FLIGHT_DIR = REPO_ROOT / "data" / "3_feature_engineered" / "flight_list"
FINAL_DATASET_PATH = REPO_ROOT / "data" / "4_final" / "model_input_merged.csv"
AIRPORTS_MASTER_PATH = REPO_ROOT / "data" / "0_master" / "airports_list.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "4_final" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"EDA outputs: {OUTPUT_DIR}")


In [ ]:
def load_csv_directory(csv_dir: Path) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(csv_dir.glob("*.csv")):
        df = pd.read_csv(csv_path)
        df["source_file"] = csv_path.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No CSV files found in {csv_dir}")
    return pd.concat(frames, ignore_index=True)


def build_missingness_report(df: pd.DataFrame, stage: str) -> pd.DataFrame:
    report = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_pct": (df.isna().mean().values * 100).round(2),
            "dtype": [str(df[col].dtype) for col in df.columns],
            "stage": stage,
        }
    )
    return report.sort_values(["missing_count", "column"], ascending=[False, True], kind="mergesort").reset_index(drop=True)


def save_csv(df: pd.DataFrame, filename: str) -> Path:
    output_path = OUTPUT_DIR / filename
    df.to_csv(output_path, index=False)
    return output_path


def hour_block_label(start_hour: int, block_size: int = 3) -> str:
    end_hour = (start_hour + block_size - 1) % 24
    return f"{start_hour:02d}:00-{end_hour:02d}:59"


raw_df = load_csv_directory(RAW_FLIGHT_DIR)
feature_df = load_csv_directory(FEATURE_FLIGHT_DIR)
final_df = pd.read_csv(FINAL_DATASET_PATH)
airports_df = pd.read_csv(AIRPORTS_MASTER_PATH)

stage_overview = pd.DataFrame(
    [
        {
            "stage": "1_raw_flight_list",
            "rows": len(raw_df),
            "columns": raw_df.drop(columns=["source_file"]).shape[1],
            "files": raw_df["source_file"].nunique(),
        },
        {
            "stage": "3_feature_engineered_flight_list",
            "rows": len(feature_df),
            "columns": feature_df.drop(columns=["source_file"]).shape[1],
            "files": feature_df["source_file"].nunique(),
        },
        {
            "stage": "4_final_model_input_merged",
            "rows": len(final_df),
            "columns": final_df.shape[1],
            "files": 1,
        },
    ]
)

missingness_report = pd.concat(
    [
        build_missingness_report(raw_df.drop(columns=["source_file"]), "1_raw_flight_list"),
        build_missingness_report(feature_df.drop(columns=["source_file"]), "3_feature_engineered_flight_list"),
        build_missingness_report(final_df, "4_final_model_input_merged"),
    ],
    ignore_index=True,
)

save_csv(stage_overview, "stage_overview.csv")
save_csv(missingness_report, "missingness_report_by_stage.csv")

print("Stage overview")
display(stage_overview)
print("Top missing columns by stage")
display(missingness_report.groupby("stage", group_keys=False).head(10))


## EDA chinh tren file merged cuoi

Tu day tro di, notebook su dung `data/4_final/model_input_merged.csv` lam bang du lieu chinh vi da ghep target, weather va airport density vao cung cap mau flight-level.


In [ ]:
analysis_df = final_df.copy()

numeric_columns = [
    "actual_turnaround",
    "scheduled_turnaround",
    "scheduled_dep_hour",
    "wind_speed",
    "visibility_meters",
    "dep_recent_index",
    "dep_recent_delay_avg",
    "is_cavok",
    "is_gust",
    "has_thunderstorm",
    "has_heavy_rain",
    "has_low_vis_fog",
]

for col in numeric_columns:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

analysis_df["std_utc"] = pd.to_datetime(analysis_df["std_utc"], errors="coerce")
analysis_df["scheduled_dep_hour"] = analysis_df["scheduled_dep_hour"].fillna(analysis_df["std_utc"].dt.hour)
analysis_df["scheduled_dep_hour"] = analysis_df["scheduled_dep_hour"].round().astype("Int64")

analysis_df["turnaround_delta"] = analysis_df["actual_turnaround"] - analysis_df["scheduled_turnaround"]

turnaround = analysis_df["actual_turnaround"].dropna()
turnaround_main = turnaround[(turnaround >= 0) & (turnaround <= 500)]

turnaround_range_summary = pd.DataFrame(
    [
        {"segment": "all_non_null", "sample_count": int(turnaround.notna().sum())},
        {"segment": "below_0", "sample_count": int((turnaround < 0).sum())},
        {"segment": "between_0_500", "sample_count": int(((turnaround >= 0) & (turnaround <= 500)).sum())},
        {"segment": "above_500", "sample_count": int((turnaround > 500).sum())},
        {"segment": "above_1440", "sample_count": int((turnaround > 1440).sum())},
    ]
)
turnaround_range_summary["share_pct"] = (turnaround_range_summary["sample_count"] / len(turnaround) * 100).round(2)

percentile_points = list(range(1, 100))
turnaround_percentile_curve = pd.DataFrame(
    {
        "percentile": percentile_points,
        "actual_turnaround_minutes": [turnaround.quantile(p / 100) for p in percentile_points],
    }
)

turnaround_percentile_table = (
    turnaround.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .rename_axis("metric")
    .reset_index(name="value")
)

turnaround_deciles = analysis_df.loc[analysis_df["actual_turnaround"].notna(), ["actual_turnaround"]].copy()
turnaround_deciles["decile"] = pd.qcut(turnaround_deciles["actual_turnaround"], q=10, duplicates="drop")
turnaround_deciles = (
    turnaround_deciles.groupby("decile", observed=False)["actual_turnaround"]
    .agg(sample_count="size", min_value="min", median_value="median", mean_value="mean", max_value="max")
    .reset_index()
)

save_csv(turnaround_range_summary, "actual_turnaround_range_summary.csv")
save_csv(turnaround_percentile_curve, "actual_turnaround_percentile_curve.csv")
save_csv(turnaround_percentile_table, "actual_turnaround_percentile_table.csv")
save_csv(turnaround_deciles, "actual_turnaround_deciles.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(turnaround_main, bins=40, color=COLORS["secondary"], edgecolor="white")
axes[0].set_title("Actual turnaround distribution (0-500 minutes)")
axes[0].set_xlabel("Actual turnaround (minutes)")
axes[0].set_ylabel("Sample count")

axes[1].plot(
    turnaround_percentile_curve["percentile"],
    turnaround_percentile_curve["actual_turnaround_minutes"],
    color=COLORS["accent"],
    linewidth=2,
)
axes[1].set_title("Actual turnaround percentile curve")
axes[1].set_xlabel("Percentile")
axes[1].set_ylabel("Minutes")
axes[1].grid(alpha=0.25)

for p in [50, 75, 90, 95, 99]:
    value = turnaround.quantile(p / 100)
    axes[1].scatter([p], [value], color=COLORS["accent"])
    axes[1].annotate(f"P{p}={value:,.1f}", (p, value), textcoords="offset points", xytext=(5, 5), fontsize=9)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "actual_turnaround_distribution_and_percentiles.png", dpi=150, bbox_inches="tight")
plt.show()

print("Turnaround range summary")
display(turnaround_range_summary)
print("Turnaround percentiles")
display(turnaround_percentile_table)
print("Turnaround deciles")
display(turnaround_deciles)


In [ ]:
high_turnaround_df = analysis_df.loc[analysis_df["actual_turnaround"] > 240].copy()

high_turnaround_hourly = (
    high_turnaround_df.loc[high_turnaround_df["scheduled_dep_hour"].notna()]
    .groupby("scheduled_dep_hour")["actual_turnaround"]
    .agg(
        high_turnaround_count="size",
        median_actual_turnaround="median",
        p90_actual_turnaround=lambda s: s.quantile(0.90),
    )
    .reset_index()
)

all_hourly = (
    analysis_df.loc[analysis_df["scheduled_dep_hour"].notna()]
    .groupby("scheduled_dep_hour")
    .size()
    .reset_index(name="all_flights")
)

high_turnaround_hourly = all_hourly.merge(high_turnaround_hourly, on="scheduled_dep_hour", how="left")
high_turnaround_hourly[["high_turnaround_count", "median_actual_turnaround", "p90_actual_turnaround"]] = high_turnaround_hourly[["high_turnaround_count", "median_actual_turnaround", "p90_actual_turnaround"]].fillna(0)
high_turnaround_hourly["high_turnaround_rate_pct"] = (high_turnaround_hourly["high_turnaround_count"] / high_turnaround_hourly["all_flights"] * 100).round(2)
high_turnaround_hourly["share_of_high_turnaround_pct"] = (high_turnaround_hourly["high_turnaround_count"] / max(len(high_turnaround_df), 1) * 100).round(2)
high_turnaround_hourly = high_turnaround_hourly.sort_values("scheduled_dep_hour").reset_index(drop=True)

high_turnaround_hourly["hour_block_start"] = (high_turnaround_hourly["scheduled_dep_hour"] // 3) * 3
high_turnaround_by_block = (
    high_turnaround_hourly.groupby("hour_block_start")
    .agg(
        all_flights=("all_flights", "sum"),
        high_turnaround_count=("high_turnaround_count", "sum"),
    )
    .reset_index()
)
high_turnaround_by_block["high_turnaround_rate_pct"] = (high_turnaround_by_block["high_turnaround_count"] / high_turnaround_by_block["all_flights"] * 100).round(2)
high_turnaround_by_block["share_of_high_turnaround_pct"] = (high_turnaround_by_block["high_turnaround_count"] / max(len(high_turnaround_df), 1) * 100).round(2)
high_turnaround_by_block["hour_block"] = high_turnaround_by_block["hour_block_start"].astype(int).map(hour_block_label)
high_turnaround_by_block = high_turnaround_by_block[["hour_block_start", "hour_block", "all_flights", "high_turnaround_count", "high_turnaround_rate_pct", "share_of_high_turnaround_pct"]]

top_high_turnaround_hours = high_turnaround_hourly.sort_values(["high_turnaround_count", "high_turnaround_rate_pct"], ascending=[False, False]).head(10)

save_csv(high_turnaround_hourly, "high_turnaround_by_hour.csv")
save_csv(high_turnaround_by_block, "high_turnaround_by_hour_block.csv")

highlight_hours = {22, 23, 0, 1, 2, 3, 4}
bar_colors = [COLORS["accent"] if int(hour) in highlight_hours else COLORS["secondary"] for hour in high_turnaround_hourly["scheduled_dep_hour"]]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(high_turnaround_hourly["scheduled_dep_hour"].astype(int), high_turnaround_hourly["high_turnaround_count"], color=bar_colors, edgecolor="white")
axes[0].set_title("Flights with actual_turnaround > 240 by scheduled hour")
axes[0].set_xlabel("Scheduled departure hour")
axes[0].set_ylabel("High-turnaround flight count")
axes[0].set_xticks(range(24))

axes[1].bar(high_turnaround_hourly["scheduled_dep_hour"].astype(int), high_turnaround_hourly["high_turnaround_rate_pct"], color=bar_colors, edgecolor="white")
axes[1].set_title("Rate of actual_turnaround > 240 within each hour")
axes[1].set_xlabel("Scheduled departure hour")
axes[1].set_ylabel("High-turnaround rate (%)")
axes[1].set_xticks(range(24))

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "high_turnaround_over_240_by_hour.png", bbox_inches="tight")
plt.show()

print("Top hours for actual_turnaround > 240 minutes")
display(top_high_turnaround_hours)
print("3-hour blocks for actual_turnaround > 240 minutes")
display(high_turnaround_by_block)


In [ ]:
delta = analysis_df["turnaround_delta"].dropna()
delta_summary = (
    delta.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .rename_axis("metric")
    .reset_index(name="value")
)

delta_direction_summary = pd.DataFrame(
    [
        {"segment": "actual_less_than_scheduled", "sample_count": int((delta < 0).sum())},
        {"segment": "actual_equal_scheduled", "sample_count": int((delta == 0).sum())},
        {"segment": "actual_greater_than_scheduled", "sample_count": int((delta > 0).sum())},
    ]
)
delta_direction_summary["share_pct"] = (delta_direction_summary["sample_count"] / len(delta) * 100).round(2)

plot_low, plot_high = delta.quantile([0.01, 0.99]).tolist()
delta_plot = delta.clip(lower=plot_low, upper=plot_high)

save_csv(delta_summary, "turnaround_delta_summary.csv")
save_csv(delta_direction_summary, "turnaround_delta_direction_summary.csv")

plt.figure(figsize=(10, 5))
plt.hist(delta_plot, bins=50, color=COLORS["highlight"], edgecolor="white")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Distribution of actual_turnaround - scheduled_turnaround")
plt.xlabel("Turnaround delta (minutes)")
plt.ylabel("Sample count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "turnaround_delta_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Turnaround delta summary")
display(delta_summary)
print("Direction summary")
display(delta_direction_summary)


In [ ]:
weather_binary_cols = [
    "is_cavok",
    "is_gust",
    "has_thunderstorm",
    "has_heavy_rain",
    "has_low_vis_fog",
]
weather_numeric_cols = ["wind_speed", "visibility_meters"]

weather_binary_rows = []
for col in weather_binary_cols:
    series = pd.to_numeric(analysis_df[col], errors="coerce")
    non_null = series.dropna()
    weather_binary_rows.append(
        {
            "feature": col,
            "non_null_count": int(non_null.shape[0]),
            "missing_count": int(series.isna().sum()),
            "positive_count": int((non_null == 1).sum()),
            "zero_count": int((non_null == 0).sum()),
            "positive_rate_non_null_pct": round((non_null.eq(1).mean() * 100), 2),
        }
    )

weather_binary_summary = pd.DataFrame(weather_binary_rows)
weather_numeric_summary = (
    analysis_df[weather_numeric_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .T.reset_index()
    .rename(columns={"index": "feature"})
)

save_csv(weather_binary_summary, "weather_binary_summary.csv")
save_csv(weather_numeric_summary, "weather_numeric_summary.csv")

print("Weather binary summary")
display(weather_binary_summary)
print("Weather numeric summary")
display(weather_numeric_summary)


In [ ]:
airport_enriched = analysis_df.merge(
    airports_df[["IATA", "Country", "Name"]],
    left_on="FROM",
    right_on="IATA",
    how="left",
)

country_counts = airport_enriched.groupby(["Country", "FROM"], dropna=False).size().reset_index(name="sample_count")
vn_hubs = country_counts[country_counts["Country"].eq("Vietnam")].nlargest(6, "sample_count")["FROM"].tolist()
intl_hubs = country_counts[country_counts["Country"].ne("Vietnam")].nlargest(6, "sample_count")["FROM"].tolist()
selected_hubs = vn_hubs + intl_hubs

hub_df = airport_enriched[airport_enriched["FROM"].isin(selected_hubs)].copy()
hub_density_summary = (
    hub_df.groupby(["Country", "FROM", "Name"], dropna=False)
    .agg(
        sample_count=("FROM", "size"),
        dep_recent_index_missing=("dep_recent_index", lambda s: int(pd.Series(s).isna().sum())),
        dep_recent_index_mean=("dep_recent_index", "mean"),
        dep_recent_index_median=("dep_recent_index", "median"),
        dep_recent_index_p90=("dep_recent_index", lambda s: pd.Series(s).quantile(0.90)),
        dep_recent_delay_avg_mean=("dep_recent_delay_avg", "mean"),
        dep_recent_delay_avg_median=("dep_recent_delay_avg", "median"),
        dep_recent_delay_avg_p90=("dep_recent_delay_avg", lambda s: pd.Series(s).quantile(0.90)),
    )
    .reset_index()
    .sort_values(["Country", "sample_count"], ascending=[True, False], kind="mergesort")
)

save_csv(hub_density_summary, "hub_departure_density_summary.csv")

plot_table = hub_density_summary.copy()
plot_table["label"] = plot_table["FROM"] + " (" + plot_table["Country"].fillna("Unknown") + ")"

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].bar(plot_table["label"], plot_table["dep_recent_index_median"], color=COLORS["secondary"], edgecolor="white")
axes[0].set_title("Median dep_recent_index by selected hub")
axes[0].set_ylabel("Median dep_recent_index")
axes[0].tick_params(axis="x", rotation=65)

axes[1].bar(plot_table["label"], plot_table["dep_recent_delay_avg_median"], color=COLORS["gold"], edgecolor="white")
axes[1].set_title("Median dep_recent_delay_avg by selected hub")
axes[1].set_ylabel("Median dep_recent_delay_avg")
axes[1].tick_params(axis="x", rotation=65)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "hub_departure_density.png", dpi=150, bbox_inches="tight")
plt.show()

print("Selected hubs")
print(selected_hubs)
display(hub_density_summary)
